# E27 - Scale up / diversify the training DATA (breadth vs depth)

**Cheapest paradigm lever.** E7 found *breadth beats depth*: training on more *distinct
series* helps, more *steps per series* hurts (correlated rows). The champion trains on only
a 6000-series subset with 10 steps each. This notebook holds the champion features + model
fixed and **varies the training pool** while keeping the **validation folds identical**.

- Breadth: `N_TRAIN in {6000 (champion), 10000, MAX}` at 10 steps/series.
- Depth control: champion 6000 series at `{10, 20}` steps/series (expect flat/worse).

Validation folds are reserved from a fixed block so every pool size is judged on the same
series. Hardened bar: `t >= 2.5` vs the 6000 champion pool. `SMOKE=1` for a quick check.

In [ ]:
import os, math
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import numpy as np
import pandas as pd

from _overnight import evaluate, paired, SMOKE, SEED, MIN_N, DATA
from _exp_fast import split_series
print("SMOKE =", SMOKE)

from _feat_lib import _feat

BASE_KEYS = ("spec_entropy",)   # base15 builder
SCALES = (60, 130, 250)         # champion multiscale
WINDOW = 250


def _subwin(w, s):
    return w if s >= len(w) else w[-s:]


def champion_feat(w, ctx):
    out = []
    for s in SCALES:
        out.extend(_feat(_subwin(w, s), ctx, keys=BASE_KEYS))
    return out

def verdict(md_, t):
    if md_ > 0 and t >= 2.5:
        return "SHIP(hard)"
    if md_ > 0 and t >= 2.0:
        return "marginal(HOLD)"
    if md_ > 0 and t >= 1.5:
        return "weak"
    return "noise"


In [ ]:
# Load a pool of a chosen train size while keeping the validation folds FIXED
# (val reserved from the END of the shuffled id list, independent of n_train).
N_FOLDS = 2 if SMOKE else 10
FOLD_SIZE = 40 if SMOKE else 400


def load_pool_n(n_train, steps):
    rng = np.random.default_rng(SEED)
    yi = pd.read_parquet(os.path.join(DATA, "y_train_index.parquet"))
    all_ids = yi.index.to_numpy().copy()
    rng.shuffle(all_ids)
    val_block = all_ids[-N_FOLDS * FOLD_SIZE:]
    pool = all_ids[:len(all_ids) - N_FOLDS * FOLD_SIZE]
    n_train = min(n_train, len(pool))
    train_ids = np.sort(pool[:n_train])
    folds = [np.sort(val_block[i * FOLD_SIZE:(i + 1) * FOLD_SIZE]) for i in range(N_FOLDS)]
    val_ids = np.concatenate(folds)
    keep = set(train_ids.tolist()) | set(val_ids.tolist())
    X = pd.read_parquet(os.path.join(DATA, "X_train.parquet"))
    X = X[X.index.get_level_values("id").isin(keep)]
    y = pd.read_parquet(os.path.join(DATA, "y_train.parquet"))
    y_val = y[y.index.get_level_values("id").isin(set(val_ids.tolist()))]
    picks = {}
    prng = np.random.default_rng(SEED + 1)
    for sid in train_ids:
        _, onl = split_series(X.loc[sid])
        L = len(onl)
        if L <= MIN_N:
            continue
        cand = np.arange(MIN_N, L)
        k = min(steps, len(cand))
        picks[sid] = prng.choice(cand, size=k, replace=False)
    return X, yi, y_val, folds, picks, len(train_ids)


yi_full = pd.read_parquet(os.path.join(DATA, "y_train_index.parquet"))
MAX_TRAIN = len(yi_full) - N_FOLDS * FOLD_SIZE
print("available training series (excl. val):", MAX_TRAIN)


In [ ]:
# Baseline: champion pool (6000 series, 10 steps). Then breadth + depth variants.
base_n = 300 if SMOKE else 6000
configs = [
    ("champion_6k_x10", base_n, 10),
    ("breadth_10k_x10", (600 if SMOKE else 10000), 10),
    ("breadth_MAX_x10", MAX_TRAIN, 10),
    ("depth_6k_x20",    base_n, 20),
]

results = {}
base_aucs = None
for name, n, steps in configs:
    X, yi, y_val, folds, picks, ntr = load_pool_n(n, steps)
    a, nf = evaluate(champion_feat, WINDOW, X, yi, y_val, picks, folds)
    results[name] = a
    if base_aucs is None:
        base_aucs = a
        print(f"{name:>16}  series={ntr:<6} rows={sum(len(v) for v in picks.values()):<6} "
              f"mean={a.mean():.4f}  (baseline)")
    else:
        md_, sd, t, _ = paired(a, base_aucs)
        print(f"{name:>16}  series={ntr:<6} rows={sum(len(v) for v in picks.values()):<6} "
              f"mean={a.mean():.4f}  delta={md_:+.4f}  t={t:+.2f}  -> {verdict(md_, t)}")
    del X


## Interpretation

If `breadth_10k`/`breadth_MAX` clears **t >= 2.5**, just retrain the champion on more series
(no code/feature change) and resubmit - the cheapest possible gain. `depth_6k_x20` is a
control: per E7 it should be flat or worse (correlated rows). If breadth is also flat, the
6000-series pool already saturates this model class.